In [156]:
import os
import pandas as pd
import numpy as np

# Read the main file and keep only columns that exist in all available CSV files.
# Always preserve the target and date columns even if they are not present in every file.
df = pd.read_csv("filtered_crmls.csv")

candidate_paths = ["filtered_crmls.csv"]
crmls_dir = "CRMLSSold"
if os.path.exists(crmls_dir):
    for filename in sorted(os.listdir(crmls_dir)):
        if filename.lower().endswith(".csv"):
            candidate_paths.append(os.path.join(crmls_dir, filename))

if len(candidate_paths) > 1:
    columns_by_file = [set(pd.read_csv(path).columns) for path in candidate_paths]
    common_columns = set.intersection(*columns_by_file)
    essential_columns = {"ClosePrice", "CloseDate"}
    common_columns.update(essential_columns.intersection(df.columns))
    if common_columns:
        df = df[[col for col in df.columns if col in common_columns]].copy()

print(df.shape)


C:\Users\saman\AppData\Local\Temp\ipykernel_20144\3843282783.py:17: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  columns_by_file = [set(pd.read_csv(path).columns) for path in candidate_paths]


(96353, 78)


In [157]:
# Keep the main price/date columns and remove only clearly unnecessary columns.
columns_to_drop = [col for col in df.columns if col.lower() in {"listprice", "originallistprice"}]
df = df.drop(columns=columns_to_drop, errors="ignore").copy()
# Drop columns that are mostly missing, but keep the target/date columns.
missing_col_ratio = df.isna().mean()
keep_cols = [col for col in df.columns if col in {"ClosePrice", "CloseDate"} or missing_col_ratio[col] <= 0.8]
df = df[keep_cols].copy()
# Report number of rows before trimming rows for missing values.
before_rows = len(df)
# Keep rows with enough data instead of dropping rows too aggressively.
min_required_non_missing = max(3, int(df.shape[1] * 0.4))
df = df.dropna(thresh=min_required_non_missing).copy()
after_rows = len(df)
print(f"Rows before filtering: {before_rows}")
print(f"Rows after filtering: {after_rows}")
print(df.shape)


Rows before filtering: 96353
Rows after filtering: 96353
(96353, 58)


In [158]:
# Apply lighter outlier filtering to avoid discarding too many rows.
for col in ["ParkingSpaces", "BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep only broadly reasonable values without being overly strict.
mask = pd.Series(True, index=df.index)

for col, low, high in [
    ("ParkingSpaces", 0, 20),
    ("BedroomsTotal", 1, 10),
    ("BathroomsTotalInteger", 1, 8),
    ("GarageSpaces", 0, 15),
]:
    if col in df.columns:
        mask &= df[col].between(low, high)

for col in ["ClosePrice", "LivingArea", "LotSizeSquareFeet", "LotSizeAcres", "YearBuilt", "DaysOnMarket"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col, low, high in [
    ("ClosePrice", 100000, 10000000),
    ("LivingArea", 400, 8000),
    ("LotSizeSquareFeet", 1000, 50000),
    ("LotSizeAcres", 0.1, 20),
    ("YearBuilt", 1800, 2026),
    ("BathroomsTotalInteger", 1, 10),
    ("BedroomsTotal", 1, 20),
    ("GarageSpaces", 0, 15),
]:
    if col in df.columns:
        mask &= df[col].between(low, high)

# Keep rows where the main target-related fields are still usable.
if "ClosePrice" in df.columns:
    mask &= df["ClosePrice"].notna()

if "CloseDate" in df.columns:
    mask &= df["CloseDate"].notna()

# Replace impossible values with NaN rather than removing rows immediately.
for col in ["ParkingSpaces", "BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces"]:
    if col in df.columns:
        df.loc[~mask, col] = np.nan

df = df.loc[mask].copy()

print(f"Rows after outlier filtering: {df.shape[0]}")
print(df.shape)


Rows after outlier filtering: 76216
(76216, 58)


In [159]:
# Split by date first so preprocessing is learned from training data only.
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")

test_file = os.path.join("CRMLSSold", "CRMLSSold202606.csv")
historical_test_df = pd.read_csv(test_file)
historical_test_df["CloseDate"] = pd.to_datetime(historical_test_df["CloseDate"], errors="coerce")

test_months = historical_test_df["CloseDate"].dt.to_period("M").dropna().unique()
test_mask = df["CloseDate"].dt.to_period("M").isin(test_months)

df_train_raw = df.loc[~test_mask].copy()
df_test_raw = df.loc[test_mask].copy()

# Identify numeric and categorical columns from the training data only.
numeric_cols = [
    col for col in df_train_raw.select_dtypes(include=[np.number]).columns.tolist()
    if col not in {"ClosePrice", "CloseDate"}
]
categorical_cols = df_train_raw.select_dtypes(include=["object"]).columns.tolist()

df = df_train_raw.copy()
df_test = df_test_raw.copy()

numeric_cols, categorical_cols


C:\Users\saman\AppData\Local\Temp\ipykernel_20144\2842308456.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train_raw.select_dtypes(include=["object"]).columns.tolist()


(['ListingKey',
  'Latitude',
  'Longitude',
  'LivingArea',
  'DaysOnMarket',
  'ListingKeyNumeric',
  'ParkingTotal',
  'LotSizeAcres',
  'YearBuilt',
  'StreetNumberNumeric',
  'BathroomsTotalInteger',
  'BedroomsTotal',
  'Stories',
  'LotSizeArea',
  'MainLevelBedrooms',
  'GarageSpaces',
  'AssociationFee',
  'LotSizeSquareFeet'],
 ['BuyerAgentAOR',
  'ListAgentAOR',
  'Flooring',
  'ViewYN',
  'PoolPrivateYN',
  'ListAgentEmail',
  'ListAgentFirstName',
  'ListAgentLastName',
  'UnparsedAddress',
  'PropertyType',
  'ListOfficeName',
  'BuyerOfficeName',
  'CoListOfficeName',
  'ListAgentFullName',
  'CoListAgentFirstName',
  'CoListAgentLastName',
  'BuyerAgentMlsId',
  'BuyerAgentFirstName',
  'BuyerAgentLastName',
  'AssociationFeeFrequency',
  'MLSAreaMajor',
  'CountyOrParish',
  'MlsStatus',
  'AttachedGarageYN',
  'PropertySubType',
  'SubdivisionName',
  'BuyerOfficeAOR',
  'ListingId',
  'City',
  'ContractStatusChangeDate',
  'PurchaseContractDate',
  'ListingContractD

In [160]:
from sklearn.preprocessing import OneHotEncoder

safe_categorical = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "PropertyType",
    "PropertySubType",
    "StateOrProvince",
    "MLSAreaMajor",
    "ElementarySchoolDistrict",
    "MiddleOrJuniorSchoolDistrict",
    "HighSchoolDistrict"
]
safe_categorical = [c for c in safe_categorical if c in df.columns]

selected_columns = [c for c in safe_categorical + numeric_cols + ["CloseDate", "ClosePrice"] if c in df.columns]
df = df[selected_columns].copy()
df = df.loc[:, ~df.columns.duplicated()]

# Fill missing values using training-only statistics.
for col in safe_categorical:
    df[col] = df[col].fillna("Unknown")

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Reduce cardinality by keeping only the top categories for each feature.
max_categories = 20
for col in safe_categorical:
    if col in df.columns:
        top_values = df[col].value_counts(dropna=False).nlargest(max_categories).index
        df[col] = df[col].where(df[col].isin(top_values), other="Other")

if safe_categorical:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded = encoder.fit_transform(df[safe_categorical])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(safe_categorical))
else:
    encoded_df = pd.DataFrame(index=df.index)


In [161]:
df[numeric_cols].isna().sum().sort_values(ascending=False)


ListingKey               0
Latitude                 0
Longitude                0
LivingArea               0
DaysOnMarket             0
ListingKeyNumeric        0
ParkingTotal             0
LotSizeAcres             0
YearBuilt                0
StreetNumberNumeric      0
BathroomsTotalInteger    0
BedroomsTotal            0
Stories                  0
LotSizeArea              0
MainLevelBedrooms        0
GarageSpaces             0
AssociationFee           0
LotSizeSquareFeet        0
dtype: int64

In [162]:
# Feature Engineering
processed_df = df.reset_index(drop=True).copy()
processed_df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
processed_df["CloseYear"] = df["CloseDate"].dt.year
processed_df["CloseMonth"] = df["CloseDate"].dt.month
processed_df["CloseYearMonth"] = df["CloseDate"].dt.to_period("M")
processed_df["BedBathRatio"] = df["BedroomsTotal"] / df["BathroomsTotalInteger"].replace(0, np.nan)
processed_df["BedroomDensity"] = df["BedroomsTotal"] / df["LivingArea"].replace(0, np.nan)
processed_df["BathroomDensity"] = df["BathroomsTotalInteger"] / df["LivingArea"].replace(0, np.nan)
processed_df["PricePerSquareFoot"] = df["ClosePrice"] / df["LivingArea"].replace(0, np.nan)
processed_df["SeasonOfSale"] = processed_df["CloseMonth"].apply(lambda x: (x % 12 + 3) // 3)  # 1: Winter, 2: Spring, 3: Summer, 4: Fall
processed_df["AgeAtSale"] = processed_df["CloseYear"] - processed_df["YearBuilt"]
processed_df["DistanceToCityCenter"] = np.sqrt((processed_df["Latitude"] - processed_df["Latitude"].mean())**2 + (processed_df["Longitude"] - processed_df["Longitude"].mean())**2)

In [163]:
# Concatenate the encoded categorical columns with the numeric columns for training.
# Note: don't include ClosePrice here to avoid duplicate columns appended below
numeric_cols = [
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "Latitude",
    "Longitude",
    "YearBuilt",
    "LotSizeSquareFeet"
]

engineered_cols = [
    "BedBathRatio",
    "BedroomDensity",
    "BathroomDensity",
    "PricePerSquareFoot",
    "SeasonOfSale",
    "AgeAtSale",
    "DistanceToCityCenter"
]
numeric_cols = numeric_cols + engineered_cols
numeric_cols = [c for c in numeric_cols if c in processed_df.columns]

if len(encoded_df) != len(processed_df):
    encoded_df = encoded_df.iloc[:len(processed_df)].copy()

if numeric_cols:
    df_encoded = pd.concat([
        processed_df[numeric_cols].reset_index(drop=True),
        encoded_df.reset_index(drop=True)
    ], axis=1)
else:
    df_encoded = encoded_df.reset_index(drop=True).copy()

if "ClosePrice" in processed_df.columns:
    df_encoded["ClosePrice"] = processed_df["ClosePrice"].to_numpy()
if "CloseDate" in processed_df.columns:
    df_encoded["CloseDate"] = processed_df["CloseDate"].to_numpy()


In [164]:
final_features = df_encoded.columns.tolist()
final_features.remove("ClosePrice")
final_features.remove("CloseDate")

In [165]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler on the training data only.
if numeric_cols and not df_encoded.empty:
    scaler = StandardScaler()
    scaled_numeric = scaler.fit_transform(df_encoded[numeric_cols])
    df_encoded[numeric_cols] = scaled_numeric


In [166]:
df_train = df_encoded.copy()

df_test = df_test_raw.copy()

# apply same feature engineering to test set
df_test["CloseDate"] = pd.to_datetime(df_test["CloseDate"], errors="coerce")
df_test["CloseYear"] = df_test["CloseDate"].dt.year
df_test["CloseMonth"] = df_test["CloseDate"].dt.month

df_test["BedBathRatio"] = df_test["BedroomsTotal"] / df_test["BathroomsTotalInteger"].replace(0, np.nan)
df_test["BedroomDensity"] = df_test["BedroomsTotal"] / df_test["LivingArea"].replace(0, np.nan)
df_test["BathroomDensity"] = df_test["BathroomsTotalInteger"] / df_test["LivingArea"].replace(0, np.nan)
df_test["PricePerSquareFoot"] = df_test["ClosePrice"] / df_test["LivingArea"].replace(0, np.nan)
df_test["SeasonOfSale"] = df_test["CloseMonth"].apply(lambda x: (x % 12 + 3) // 3)
df_test["AgeAtSale"] = df_test["CloseYear"] - df_test["YearBuilt"]

df_test["DistanceToCityCenter"] = np.sqrt(
    (df_test["Latitude"] - df_train["Latitude"].mean())**2 +
    (df_test["Longitude"] - df_train["Longitude"].mean())**2
)


# Prepare the test set with the same categorical and numeric transformations
if "CloseDate" in df_test.columns:
    df_test["CloseDate"] = pd.to_datetime(df_test["CloseDate"], errors="coerce")


for col in numeric_cols:
    if col not in df_test.columns:
        df_test[col] = np.nan

for col in safe_categorical:
    if col not in df_test.columns:
        df_test[col] = "Unknown"

for col in final_features:
    if col not in df_test.columns:
        df_test[col] = np.nan

encoded_test = encoder.transform(df_test[safe_categorical])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(safe_categorical))

# Build full test matrix BEFORE selecting final_features
df_test_full = pd.concat([
    df_test[numeric_cols].reset_index(drop=True),
    encoded_test_df.reset_index(drop=True)
], axis=1)

df_test_full["ClosePrice"] = df_test["ClosePrice"].to_numpy()
df_test_full["CloseDate"] = df_test["CloseDate"].to_numpy()

# Now enforce final_features
for col in final_features:
    if col not in df_test_full.columns:
        df_test_full[col] = np.nan

df_test_processed = df_test_full[final_features + ["ClosePrice", "CloseDate"]].copy()


# Apply the scaler fitted on training data.
if numeric_cols and not df_test_processed.empty:
    test_numeric = df_test_processed[numeric_cols]
    scaled_test_numeric = scaler.transform(test_numeric)
    df_test_processed[numeric_cols] = scaled_test_numeric

project_dir = os.getcwd()
os.makedirs(project_dir, exist_ok=True)

output_path = os.path.join(project_dir, "cleaned_preprocessed.csv")
train_output_path = os.path.join(project_dir, "df_train.csv")
test_output_path = os.path.join(project_dir, "df_test.csv")
processed_test_output_path = os.path.join(project_dir, "df_test_processed.csv")

df_encoded.to_csv(output_path, index=False)
df_train.to_csv(train_output_path, index=False)
df_test.to_csv(test_output_path, index=False)
df_test_processed.to_csv(processed_test_output_path, index=False)

print(f"Saved {len(df_train)} training rows and {len(df_test)} test rows")


C:\Users\saman\AppData\Local\Temp\ipykernel_20144\4098328339.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test[col] = np.nan


Saved 66052 training rows and 10164 test rows
